## Download Packages

In [82]:
from pathlib import Path
import numpy as np
from pipeline_functions import *
import torch

## Choose your Subject and Character

In [83]:
subject =  5 # 1-55
character = 'B' 

## Behind the Scenes Work

In [84]:
base_dir = Path.cwd()
file_path = base_dir / f"datasets/Won2022_BIDS/.mat_files/s{subject:02d}.mat"

print(file_path)


c:\Users\crims\Desktop\Senior Design Code\VirtualKeyboard\datasets\Won2022_BIDS\.mat_files\s05.mat


In [85]:
character_data = fn_preprocess.load_subject_character_data(file_path, letter=character)

Found B in test set 0
Randomly selected test set 0 for letter B
Randomly selected position 2 for letter B in test set 0


In [86]:
print(len(np.where((character_data['markers_seq'] >= 1) & (character_data['markers_seq'] <= 12))[0]))

180


In [87]:
X, y, time, char = fn_preprocess.preprocess_one_character(character_data, target_char=character)

print(X.shape)
print(y.shape)

(8, 307, 180)
(180,)


In [88]:
X, y = fn_feature_extraction.extractFeatures(X, y, k=1, factor=3, t_min=200, t_max=400, norm_type='std')

X shape: (180, 8, 307)
Shape after feature extraction: (180, 8, 34)


In [89]:
X_spikes = delta_encoding.delta_encode(X, 0.072)
print(X_spikes.shape)

(180, 34, 8)


In [90]:
X_spikes = torch.from_numpy(X_spikes).float()
y = y.astype(int)

In [91]:
weights_path = base_dir / "model_weights/2hl12864_th072_win2_4_f3_weights.pth"
print(weights_path)

snn = SNNModule.createSNN(8, [128, 64], betas=[0.95, 0.95, 0.95], thresholds=[1, 1, 1])

# load in weights
weights = torch.load(weights_path, weights_only=True)
snn.load_state_dict(weights)

snn.eval()
with torch.no_grad():
    spk_out, mem_out = snn(X_spikes, batch_first=True)

c:\Users\crims\Desktop\Senior Design Code\VirtualKeyboard\model_weights\2hl12864_th072_win2_4_f3_weights.pth


In [92]:
selected_char, row_idx, col_idx, row_total, col_total = Characterselection.p300_speller_cycle_prob(spk_out, y)

## Predicted Character Output

In [93]:
print(f"Selected character: {selected_char}")
print(f"Correct character: {character}")

Selected character: B
Correct character: B
